In [1]:
#turn on breakpoints?
%pdb on

Automatic pdb calling has been turned ON


In [2]:
!git init
!git remote add origin https://github.com/sgarnell/archive

Reinitialized existing Git repository in /home/ffbo/ffbo/.git/
fatal: remote origin already exists.


In [3]:
!git remote -v

origin	https://github.com/sgarnell/archive.git (fetch)
origin	https://github.com/sgarnell/archive.git (push)


In [ ]:
# Stage the notebook file
!git add 'PC_Latex_Generator_v1.ipynb'

# Commit the changes
!git commit -m "Auto commit from Jupyter for {notebook_name}"

# Push to GitHub
!git push origin fbl2-main  # Replace 'main' with 'master' if needed

In [ ]:
import os
import json
import re

def load_files(dot_file, nt_file, motif_file):
    with open(dot_file, "r") as f:
        dot_lines = f.readlines()
    with open(nt_file, "r") as f:
        nt_dict = json.load(f)
    with open(motif_file, "r") as f:
        motif_dict = json.load(f)
    return dot_lines, nt_dict, motif_dict

def get_polarity(neuron, nt_dict, polarity_dict):
    nts = nt_dict.get(neuron, {})
    for nt in nts:
        sign = polarity_dict.get(nt.lower())
        if sign is not None:
            return "+" if sign == 1 else "-"
    return "?"

def extract_edges(dot_lines):
    edges = []
    for line in dot_lines:
        match = re.search(r'"(.+?)" -> "(.+?)" 

\[label="Zscore = ([0-9.]+)"', line)
        if match:
            src, mid, zscore = match.groups()
            edges.append((src, mid, float(zscore)))
    return edges

def build_neuron_equations(edges, nt_dict, polarity_dict):
    synapse_map = {}
    for src, mid, z1 in edges:
        if "--" in mid:
            neuron_a, neuron_b = mid.split("--")
            synapse_map[mid] = {"from": src, "to": neuron_b, "z1": z1}
        else:
            for key in synapse_map:
                if mid == synapse_map[key]["to"]:
                    synapse_map[key]["z2"] = z1

    equations = []
    for key, val in synapse_map.items():
        if "z2" not in val:
            continue
        z = round(val["z1"] * val["z2"], 2)
        polarity = get_polarity(val["from"], nt_dict, polarity_dict)
        eq = (
            f"\\mu_{{{val['to']}}}^{{(P)}} = "
            f"\\bm{{W}}_{{{val['to']} \\leftarrow {val['from']}}}^{{({polarity})}}(z = {z}) "
            f"\\cdot \\mu_{{{val['from']}}}^{{(A)}}"
        )
        equations.append(eq)
    return equations

def build_motif_equations(motif_dict):
    equations = []
    for motif_name, motif in motif_dict.items():
        motif_type = motif.get("motifType", "unknown")
        A = motif["neuron_A"]
        B = motif["neuron_B"]
        z_A = motif.get("Z_A", 1.0)
        z_B = motif.get("Z_B", 1.0)
        NT_A = motif.get("NT_A", "?")
        NT_B = motif.get("NT_B", "?")

        polarity_A = "+" if NT_A.lower() in ["acetylcholine", "dopamine", "octopamine"] else "-"
        polarity_B = "+" if NT_B.lower() in ["acetylcholine", "dopamine", "octopamine"] else "-"

        motif_label = f"{motif_type.upper()}_{{{A},{B}}}"

        eq = (
            f"\\mu_{{{motif_label}}}^{{(P)}} = "
            f"\\bm{{W}}_{{{motif_label} \\leftarrow {A}}}^{{({polarity_A})}}(z = {z_A}) "
            f"\\cdot \\mu_{{{A}}}^{{(A)}} + "
            f"\\bm{{W}}_{{{motif_label} \\leftarrow {B}}}^{{({polarity_B})}}(z = {z_B}) "
            f"\\cdot \\mu_{{{B}}}^{{(A)}}"
        )
        equations.append(eq)

        # Optional: internal dynamics
        if motif_type == "wilson-cowan":
            equations.append(
                f"\\tau \\dot{{\\mu}}_{{{A}}} = -\\mu_{{{A}}} + f(\\mu_{{{B}}})"
            )
            equations.append(
                f"\\tau \\dot{{\\mu}}_{{{B}}} = -\\mu_{{{B}}} + g(\\mu_{{{A}}})"
            )

    return equations

def write_latex(equations, filename):
    with open(filename, "w") as f:
        f.write("\\section*{Predictive Coding Equations}\n\n")
        for eq in equations:
            f.write("\\begin{equation}\n")
            f.write(eq + "\n")
            f.write("\\end{equation}\n\n")

def main():
    folder = "PC_LatexGen"
    dot_file = os.path.join(folder, "MBON03_prePost_merged_filtered.gv.txt")
    nt_file = os.path.join(folder, "neuronNTdict.json")
    motif_file = os.path.join(folder, "motifs.json")

    polarity_dict = {
        "acetylcholine": 1,
        "glutamate": 0,
        "gaba": 0,
        "dopamine": 1,
        "octopamine": 1,
        "serotonin": 0
    }

    dot_lines, nt_dict, motif_dict = load_files(dot_file, nt_file, motif_file)
    edges = extract_edges(dot_lines)
    neuron_eqs = build_neuron_equations(edges, nt_dict, polarity_dict)
    motif_eqs = build_motif_equations(motif_dict)

    all_eqs = neuron_eqs + motif_eqs
    output_path = os.path.expanduser("~/predictive_coding_equations.tex")
    write_latex(all_eqs, output_path)
    print(f"LaTeX file written to: {output_path}")

if __name__ == "__main__":
    main()
